# Surrogate Model Workflow Notebook

This notebook shows the full workflow for the repository:

1. **Data generation**
2. **Model training**
3. **Inverse optimisation**

The workflow is intentionally simple. 
Each stage directly imports and runs the relevant module.

Repository structure expected:

```text
surrogate_model_clean/
├── configs/
│   ├── data_config.py
│   ├── training_config.py
│   └── optimiser_config.py
├── src/
│   ├── data_generation.py
│   ├── model_training.py
│   └── optimiser.py
└── notebooks/
```

Run this notebook from anywhere, but update `REPO_DIR` in the next cell.


## 0. Setup paths

Set `REPO_DIR` to the root of your local repository.

Example:

```python
REPO_DIR = Path("/home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean")
```

The notebook then adds `src/` and `configs/` to `sys.path`, so Python can import your modules directly.


In [1]:
from pathlib import Path
import sys
import importlib
import json
import os

# ---------------------------------------------------------------------
# CHANGE THIS PATH IF NEEDED
# ---------------------------------------------------------------------
REPO_DIR = Path("/home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean")

SRC_DIR = REPO_DIR / "src"
CONFIG_DIR = REPO_DIR / "configs"

# Add source and config folders to import path
for p in [SRC_DIR, CONFIG_DIR]:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

print("REPO_DIR   :", REPO_DIR)
print("SRC_DIR    :", SRC_DIR)
print("CONFIG_DIR :", CONFIG_DIR)

assert REPO_DIR.exists(), f"REPO_DIR does not exist: {REPO_DIR}"
assert SRC_DIR.exists(), f"SRC_DIR does not exist: {SRC_DIR}"
assert CONFIG_DIR.exists(), f"CONFIG_DIR does not exist: {CONFIG_DIR}"


REPO_DIR   : /home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean
SRC_DIR    : /home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/src
CONFIG_DIR : /home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/configs


## 1. Choose experiment settings

Use small values first to test that the full workflow works.

After the notebook runs correctly, increase:

- `NODES`
- `N_SAMPLES`
- `DATA_BATCH_SIZE`
- `TRAINING_BATCH_SIZE`
- `NUM_EPOCHS`
- `OPTIMISATION_STEPS`

For a quick notebook test, start with `NODES = 4`.


In [2]:
# ---------------------------------------------------------------------
# Main experiment settings
# ---------------------------------------------------------------------
NODES = 4
DIMENSIONS = 2
TARGET_NAME = "GHZ"          # Options usually include: "GHZ", "W", "LINEAR_CLUSTER", "SINGLE"
MODEL_NAME = "HNN"           # "HNN" or "FNN"

# ---------------------------------------------------------------------
# Data generation settings
# ---------------------------------------------------------------------
N_SAMPLES = 1000
DATA_BATCH_SIZE = 100
DATA_SEED = 59
SHARD_SIZE = 500
NORMED_DATA = False

DATA_DIR = REPO_DIR / f"data/node{NODES}_notebook_test"

# ---------------------------------------------------------------------
# Training settings
# ---------------------------------------------------------------------
HIDDEN_DIM = 400              # For HNN: integer. For FNN: usually tuple/list, e.g. (400, 400, 400)
TRAINING_BATCH_SIZE = 100
NUM_EPOCHS = 20
PATIENCE = 10
LOSS_NAME = "mae"

MODEL_ROOT = REPO_DIR / "Models_notebook_test"
RUN_NAME = f"{MODEL_NAME}_{NODES}_{TARGET_NAME}_notebook_test"

# ---------------------------------------------------------------------
# Optimisation settings
# ---------------------------------------------------------------------
OPTIMISATION_STEPS = 20
OPTIMISER_SAMPLES = 5
OPTIMISER_RESULTS_ROOT = REPO_DIR / "optimiser_notebook_results"
OPTIMISER_FOLDER_NAME = f"{TARGET_NAME}_{NODES}_quick_test"

print("Data directory:", DATA_DIR)
print("Model run name:", RUN_NAME)
print("Optimiser output:", OPTIMISER_RESULTS_ROOT / OPTIMISER_FOLDER_NAME)


Data directory: /home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/data/node4_notebook_test
Model run name: HNN_4_GHZ_notebook_test
Optimiser output: /home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/optimiser_notebook_results/GHZ_4_quick_test


## 2. Generate data

This cell directly imports:

```python
from data_generation import generate_dataset
```

The function returns the output folder when `save_to_file=True`.

The generated data is saved as shard files inside `DATA_DIR`.


In [3]:
from data_generation import generate_dataset

out_dir = generate_dataset(
    vertices=NODES,
    dimensions=DIMENSIONS,
    n_samples=N_SAMPLES,
    batch_size=DATA_BATCH_SIZE,
    seed=DATA_SEED,
    save_to_file=True,
    out_dir_path=str(DATA_DIR),
    shard_size=SHARD_SIZE,
    normed_data=NORMED_DATA,
)

out_dir = Path(out_dir)

print("Generated data folder:", out_dir)
print("Files:")
for f in sorted(out_dir.glob("*")):
    print(" -", f.name)


2026-05-29 14:52:57,556 | INFO | Log file: /home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/data/node4_notebook_test/logs/data_generation_20260529_145257.log
2026-05-29 14:52:57,557 | INFO | === Random Dense Graph Dataset Generation ===


ERROR:2026-05-29 14:52:58,545:jax._src.xla_bridge:475: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/bo48god/projects/deep_dreaming/env_jax/lib/python3.12/site-packages/jax/_src/xla_bridge.py", line 473, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/bo48god/projects/deep_dreaming/env_jax/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/bo48god/projects/deep_dreaming/env_jax/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to load the CU

2026-05-29 14:52:58,658 | INFO | JAX devices: [CpuDevice(id=0)]
2026-05-29 14:52:58,659 | INFO | vertices=4, dimensions=2
2026-05-29 14:52:58,660 | INFO | n_samples=1,000, batch_size=100
2026-05-29 14:52:58,661 | INFO | seed=59
2026-05-29 14:52:58,662 | INFO | normed_data=False
2026-05-29 14:52:58,663 | INFO | graph regime=Level 5 random dense weights in [-1, 1]
2026-05-29 14:52:58,664 | INFO | Precomputing PyTheus perfect-matching catalog...
2026-05-29 14:52:58,672 | INFO | num_edges=24
2026-05-29 14:52:58,674 | INFO | n_kets=16
2026-05-29 14:52:58,676 | INFO | catalog tensor shape=(16, 3, 2)
2026-05-29 14:52:58,678 | INFO | catalog time=0.01s
2026-05-29 14:52:58,679 | INFO | Transferring catalog to GPU...
2026-05-29 14:52:58,918 | INFO | JIT warmup...
2026-05-29 14:52:59,205 | INFO | Generating data batches...
2026-05-29 14:52:59,242 | INFO | Saved shard 00000 -> /home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/data/node4_notebook

## 3. Merge data shards

Training expects a single `.npz` file.

This cell merges the generated shards into:

```text
dataset_merged.npz
```

inside the same data folder.


In [4]:
from data_generation_utils import merge_shards_to_npz

merged_path = merge_shards_to_npz(
    out_dir=out_dir,
    output_name="dataset_merged.npz",
)

merged_path = Path(merged_path)

print("Merged dataset:", merged_path)
print("Exists:", merged_path.exists())


Merged dataset: /home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/data/node4_notebook_test/dataset_merged.npz
Exists: True


## 4. Inspect merged dataset

This sanity check verifies the shapes of `X` and `Y`.

Expected dimensions:

```python
input_dim = 2 * NODES * (NODES - 1)
out_dim   = 2 ** NODES
```

For `NODES = 4`:

```python
input_dim = 24
out_dim = 16
```


In [5]:
import numpy as np

with np.load(merged_path) as data:
    print("Keys:", list(data.keys()))
    for key in data.keys():
        arr = data[key]
        print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

INPUT_DIM = 2 * NODES * (NODES - 1)
OUT_DIM = 2 ** NODES

print("Expected input_dim:", INPUT_DIM)
print("Expected out_dim   :", OUT_DIM)


Keys: ['weights', 'amps']
weights: shape=(1000, 24), dtype=float32
amps: shape=(1000, 16), dtype=float32
Expected input_dim: 24
Expected out_dim   : 16


## 5. Train the surrogate model

Your current `model_training.py` reads `TRAINING_CONFIG` at import time.

Because of that, the correct notebook pattern is:

```python
import training_config
training_config.TRAINING_CONFIG.update({...})

import model_training
importlib.reload(model_training)

run_dir = model_training.train_surrogate_model()
```

The `reload()` is important.  
It makes sure `model_training.py` sees the updated config values.


In [6]:
import training_config

training_config.TRAINING_CONFIG.update({
    # -----------------------------------------------------------------
    # Experiment identity
    # -----------------------------------------------------------------
    "NODES": NODES,
    "DIMENSIONS": DIMENSIONS,
    "DATE": "notebook_test",

    # -----------------------------------------------------------------
    # Model
    # -----------------------------------------------------------------
    "MODEL_NAME": MODEL_NAME,
    "HIDDEN_DIM": HIDDEN_DIM,

    # -----------------------------------------------------------------
    # Dataset
    # -----------------------------------------------------------------
    "DATA_PATH": str(merged_path),
    "DATA_SIZE": None,        # None = use full dataset

    # -----------------------------------------------------------------
    # Output normalization
    # -----------------------------------------------------------------
    # Keep this consistent with the data and optimiser settings.
    "NORMALIZE_MODEL_OUTPUT": False,

    # -----------------------------------------------------------------
    # Dataset split
    # -----------------------------------------------------------------
    "TRAIN_SPLIT": 0.8,
    "VAL_SPLIT": 0.1,
    "TEST_SPLIT": 0.1,

    # -----------------------------------------------------------------
    # Optimizer and training schedule
    # -----------------------------------------------------------------
    "LEARNING_RATE": 1e-3,
    "LR_AFTER_DECAY": 1e-5,
    "LR_DECAY_UNTIL_EPOCH": max(NUM_EPOCHS, 1),

    "BATCH_SIZE": TRAINING_BATCH_SIZE,
    "NUM_EPOCHS": NUM_EPOCHS,
    "PATIENCE": PATIENCE,
    "TOLERANCE": 1e-7,
    "INIT_KEY": 159,

    # -----------------------------------------------------------------
    # Resume training
    # -----------------------------------------------------------------
    "RESUME_FULL_STATE": False,
    "CKPT_DIR_RESTORE": "",

    # -----------------------------------------------------------------
    # Output folders
    # -----------------------------------------------------------------
    "ROOT_FOLDER": str(MODEL_ROOT),
    "RUN_NAME": RUN_NAME,
    "LOSS_NAME": LOSS_NAME,
})

print(json.dumps(training_config.TRAINING_CONFIG, indent=2))


{
  "NODES": 4,
  "DIMENSIONS": 2,
  "DATE": "notebook_test",
  "MODEL_NAME": "HNN",
  "HIDDEN_DIM": 400,
  "DATA_PATH": "/home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/data/node4_notebook_test/dataset_merged.npz",
  "DATA_SIZE": null,
  "NORMALIZE_MODEL_OUTPUT": false,
  "TRAIN_SPLIT": 0.8,
  "VAL_SPLIT": 0.1,
  "TEST_SPLIT": 0.1,
  "LEARNING_RATE": 0.001,
  "LR_AFTER_DECAY": 1e-05,
  "LR_DECAY_UNTIL_EPOCH": 20,
  "BATCH_SIZE": 100,
  "NUM_EPOCHS": 20,
  "PATIENCE": 10,
  "TOLERANCE": 1e-07,
  "INIT_KEY": 159,
  "RESUME_FULL_STATE": false,
  "CKPT_DIR_RESTORE": "",
  "ROOT_FOLDER": "/home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/Models_notebook_test",
  "RUN_NAME": "HNN_4_GHZ_notebook_test",
  "LOSS_NAME": "mae"
}


In [7]:
import model_training
importlib.reload(model_training)

run_dir = model_training.train_surrogate_model()
run_dir = Path(run_dir)

print("Training run directory:", run_dir)
print("Files:")
for f in sorted(run_dir.glob("*")):
    print(" -", f.name)


Training run directory: /home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/Models_notebook_test/run_HNN_4_GHZ_notebook_test
Files:
 - checkpoints
 - model_info.json
 - params.msgpack
 - params_snapshots
 - plots
 - run_log.txt
 - test_metrics.npz


## 6. Locate trained model parameters

The optimiser needs the trained model parameter file:

```text
params.msgpack
```

This cell checks that the file exists.


In [8]:
params_path = run_dir / "params.msgpack"

print("Params path:", params_path)
print("Exists:", params_path.exists())

assert params_path.exists(), (
    "params.msgpack was not found. Check whether training finished successfully "
    "and whether save_params_msgpack() was called inside model_training.py."
)


Params path: /home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/Models_notebook_test/run_HNN_4_GHZ_notebook_test/params.msgpack
Exists: True


## 7. Run inverse optimisation

This directly imports:

```python
from optimiser import run_optimisation
```

Unlike `model_training.py`, the optimiser already accepts a config dictionary:

```python
run_optimisation(CFG_OPT)
```

So there is no need to edit `configs/optimiser_config.py`.


In [9]:
from optimiser import run_optimisation

CFG_OPT = {
    "description": "notebook inverse optimisation test",

    # -----------------------------------------------------------------
    # Main setup
    # -----------------------------------------------------------------
    "n": NODES,
    "dimensions": DIMENSIONS,
    "target_name": TARGET_NAME,
    "model_type": MODEL_NAME,

    # -----------------------------------------------------------------
    # Initial samples for optimisation
    # -----------------------------------------------------------------
    # True  = generate fresh random samples.
    # False = load samples from conditioned_data_path.
    "generate_data": True,
    "conditioned_data_path": "",
    "max_initial_samples": None,

    # Used when generate_data=True
    "data_samples": OPTIMISER_SAMPLES,
    "data_batch_size": OPTIMISER_SAMPLES,
    "data_seed": 4,
    "low_fidelity_threshold": 0.999,
    "save_generated_data": False,
    "data_out_dir": str(REPO_DIR / "data/optimiser_generated"),
    "normed_data": True,
    "generation_gpu_batch_size": OPTIMISER_SAMPLES,
    "data_shard_size": OPTIMISER_SAMPLES,

    # -----------------------------------------------------------------
    # Trained surrogate model
    # -----------------------------------------------------------------
    # For HNN use an integer, e.g. 400 or 15000.
    # For FNN use a tuple/list, e.g. (400, 400, 400).
    "architecture": HIDDEN_DIM,
    "model_path": str(params_path),
    "normalize_model_output": False,

    # Derived dimensions
    "input_dim": 2 * NODES * (NODES - 1),
    "out_dim": 2 ** NODES,

    # -----------------------------------------------------------------
    # Optimisation objective
    # loss = 1 - fidelity + lambda_l1 * L1
    # -----------------------------------------------------------------
    "lambda_l1": 1e-3,

    # -----------------------------------------------------------------
    # Optimisation schedule
    # -----------------------------------------------------------------
    "seed": 46,
    "num_steps": OPTIMISATION_STEPS,
    "early_stop_nn_fid": 0.9999,

    "learning_rate": 1e-2,
    "min_learning_rate": 1e-6,
    "lr_decay_steps": OPTIMISATION_STEPS,
    "lr_exponent": 1.0,

    # Keep graph weights inside the training domain.
    "clip_min": -1.0,
    "clip_max": 1.0,

    # Logging and verification
    "print_every": 1,
    "verify_every": 1,

    # Store step vectors only when debugging. Large runs can create huge JSON files.
    "store_step_vectors": False,

    # Pruning
    "prune_fid_tolerance": 1e-4,
    "prune_thresholds": [1e-5, 1e-4, 1e-3, 1e-2, 1e-1],

    # Output
    "results_root": str(OPTIMISER_RESULTS_ROOT),
    "folder_name": OPTIMISER_FOLDER_NAME,

    # Metadata
    "date": "notebook",
    "k_value": 0,
    "noise": 0,
    "CPU_GPU": "GPU",
}

print(json.dumps(CFG_OPT, indent=2))


{
  "description": "notebook inverse optimisation test",
  "n": 4,
  "dimensions": 2,
  "target_name": "GHZ",
  "model_type": "HNN",
  "generate_data": true,
  "conditioned_data_path": "",
  "max_initial_samples": null,
  "data_samples": 5,
  "data_batch_size": 5,
  "data_seed": 4,
  "low_fidelity_threshold": 0.999,
  "save_generated_data": false,
  "data_out_dir": "/home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/data/optimiser_generated",
  "normed_data": true,
  "generation_gpu_batch_size": 5,
  "data_shard_size": 5,
  "architecture": 400,
  "model_path": "/home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/Models_notebook_test/run_HNN_4_GHZ_notebook_test/params.msgpack",
  "normalize_model_output": false,
  "input_dim": 24,
  "out_dim": 16,
  "lambda_l1": 0.001,
  "seed": 46,
  "num_steps": 20,
  "early_stop_nn_fid": 0.9999,
  "learning_rate": 0.01,
  "min_learning_rate": 1e-06,

In [10]:
result_dir = run_optimisation(CFG_OPT)
result_dir = Path(result_dir)

print("Optimisation result directory:", result_dir)
print("Files:")
for f in sorted(result_dir.glob("*")):
    print(" -", f.name)


2026-05-29 14:53:36,053 | INFO | === Random Dense Graph Dataset Generation ===
2026-05-29 14:53:36,056 | INFO | JAX devices: [CpuDevice(id=0)]
2026-05-29 14:53:36,057 | INFO | vertices=4, dimensions=2
2026-05-29 14:53:36,059 | INFO | n_samples=5, batch_size=5
2026-05-29 14:53:36,061 | INFO | seed=4
2026-05-29 14:53:36,062 | INFO | normed_data=True
2026-05-29 14:53:36,063 | INFO | graph regime=Level 5 random dense weights in [-1, 1]
2026-05-29 14:53:36,065 | INFO | Precomputing PyTheus perfect-matching catalog...
2026-05-29 14:53:36,070 | INFO | num_edges=24
2026-05-29 14:53:36,071 | INFO | n_kets=16
2026-05-29 14:53:36,072 | INFO | catalog tensor shape=(16, 3, 2)
2026-05-29 14:53:36,073 | INFO | catalog time=0.01s
2026-05-29 14:53:36,075 | INFO | Transferring catalog to GPU...
2026-05-29 14:53:36,077 | INFO | JIT warmup...
2026-05-29 14:53:36,314 | INFO | Generating data batches...
2026-05-29 14:53:36,336 | INFO | batch     1 |          5/5 (100.0%) | 0.0s | 249 samples/s
2026-05-29 14

## 8. Read optimisation summary files

The optimiser writes several JSON and plot files.  
This cell prints any JSON summaries found in the result folder.


In [11]:
json_files = sorted(result_dir.glob("*.json"))

print("JSON files found:")
for jf in json_files:
    print(" -", jf.name)

for jf in json_files:
    print("\n" + "=" * 80)
    print(jf.name)
    print("=" * 80)
    try:
        with open(jf, "r") as f:
            content = json.load(f)
        print(json.dumps(content, indent=2)[:5000])
        if len(json.dumps(content)) > 5000:
            print("\n... truncated ...")
    except Exception as e:
        print("Could not read JSON:", e)


JSON files found:
 - best_graph_solution.json
 - cfg.json
 - optimisation_summary.json
 - zero_state_samples.json

best_graph_solution.json
{
  "sample_id": 1,
  "num_edges": 22,
  "graph_vector": [
    0.0,
    -0.012760331854224205,
    0.0,
    0.9171645641326904,
    -0.30012521147727966,
    -0.5524576902389526,
    0.04417400434613228,
    0.2823418378829956,
    0.8782141208648682,
    0.16403159499168396,
    -0.4643332064151764,
    0.8595494031906128,
    -0.016549430787563324,
    0.3516017198562622,
    -0.047922179102897644,
    -0.5660399198532104,
    0.38510408997535706,
    0.5412609577178955,
    -0.6184229850769043,
    -0.0801679715514183,
    -0.2763698697090149,
    -0.6585403680801392,
    -0.5572966933250427,
    0.9250147938728333
  ],
  "nn_state": [
    2.0541324615478516,
    -0.7373837828636169,
    -1.4126838445663452,
    0.4427897036075592,
    0.7035601139068604,
    -2.861515522003174,
    4.192122936248779,
    -0.17718517780303955,
    1.306625008583

# Notes for larger runs

After the quick test works, change these values:

```python
NODES = 6 or 8
N_SAMPLES = larger number
DATA_BATCH_SIZE = larger batch size
SHARD_SIZE = larger shard size
HIDDEN_DIM = your real architecture
NUM_EPOCHS = real training epochs
TRAINING_BATCH_SIZE = real training batch size
OPTIMISATION_STEPS = 10000 or your target value
OPTIMISER_SAMPLES = number of starting samples
```

For large data generation and training:

- Run on the cluster/GPU.
- Keep `data/`, `Models_*`, and `optimiser_*` out of GitHub.
- Use notebook values only for testing.
- Move final production settings back into config files or Slurm scripts.


# Minimal workflow summary

The full notebook workflow is:

```python
from data_generation import generate_dataset
out_dir = generate_dataset(...)

from data_generation_utils import merge_shards_to_npz
merged_path = merge_shards_to_npz(...)

import training_config
training_config.TRAINING_CONFIG.update({...})

import model_training
importlib.reload(model_training)
run_dir = model_training.train_surrogate_model()

from optimiser import run_optimisation
result_dir = run_optimisation(CFG_OPT)
```
